# SASRec Delta-Start Time-Aware BPI2012 Colab Train Combo (`refine_ml50_do035` baseline)

Colab notebook for comparing two `delta_start_seconds` time-aware variants on top of `refine_ml50_do035`:
- `delta_start_seconds + 9-bucket`
- `delta_start_seconds + continuous`

Both are evaluated under `NDCG@10` and `NDCG@5` model-selection criteria.


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
BUCKET_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10'
BUCKET_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5'
CONT_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10'
CONT_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('BUCKET_NDCG10_OUTPUT_DIR:', BUCKET_NDCG10_OUTPUT_DIR)
print('BUCKET_NDCG5_OUTPUT_DIR:', BUCKET_NDCG5_OUTPUT_DIR)
print('CONT_NDCG10_OUTPUT_DIR:', CONT_NDCG10_OUTPUT_DIR)
print('CONT_NDCG5_OUTPUT_DIR:', CONT_NDCG5_OUTPUT_DIR)


In [ ]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$BUCKET_NDCG10_OUTPUT_DIR"
!mkdir -p "$BUCKET_NDCG5_OUTPUT_DIR"
!mkdir -p "$CONT_NDCG10_OUTPUT_DIR"
!mkdir -p "$CONT_NDCG5_OUTPUT_DIR"


In [ ]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


In [ ]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


In [ ]:
!pip install -r requirements_colab.txt


In [ ]:
!ls "$DATA_DIR"


In [ ]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Time-aware variants:
- `delta_start_seconds + 9-bucket`
- `delta_start_seconds + continuous`


## Check baseline runs


In [ ]:
from pathlib import Path

baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

for label, output_dir in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR)),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR)),
]:
    print('=' * 80)
    print(label)
    for run_name in baseline_runs:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


## Check planned new runs


In [ ]:
planned_bucket_ndcg10 = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
planned_bucket_ndcg5 = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
planned_cont_ndcg10 = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]
planned_cont_ndcg5 = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]

for label, output_dir, run_names in [
    ('Bucket NDCG@10', Path(BUCKET_NDCG10_OUTPUT_DIR), planned_bucket_ndcg10),
    ('Bucket NDCG@5', Path(BUCKET_NDCG5_OUTPUT_DIR), planned_bucket_ndcg5),
    ('Continuous NDCG@10', Path(CONT_NDCG10_OUTPUT_DIR), planned_cont_ndcg10),
    ('Continuous NDCG@5', Path(CONT_NDCG5_OUTPUT_DIR), planned_cont_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


## Train `delta_start + 9-bucket` for `NDCG@10`


### timeaware_dstart_refine_ml50_do035_b9_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_refine_ml50_do035_b9_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_refine_ml50_do035_b9_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Train `delta_start + continuous` for `NDCG@10`


### timeaware_dstart_conti_refine_ml50_do035_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_conti_refine_ml50_do035_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_conti_refine_ml50_do035_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Train `delta_start + 9-bucket` for `NDCG@5`


### timeaware_dstart_refine_ml50_do035_b9_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_refine_ml50_do035_b9_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_refine_ml50_do035_b9_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_refine_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Train `delta_start + continuous` for `NDCG@5`


### timeaware_dstart_conti_refine_ml50_do035_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_conti_refine_ml50_do035_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware_dstart_conti_refine_ml50_do035_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_conti_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_start_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Rebuild result tables


In [ ]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'time_encoding': config.get('time_encoding'),
            'time_delta_column': config.get('time_delta_column'),
            'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
            'time_feature_dim': config.get('time_feature_dim'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## NDCG@10 comparison summary


In [ ]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
bucket_runs = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
continuous_runs = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
bucket_df = rebuild_df(BUCKET_NDCG10_OUTPUT_DIR)
continuous_df = rebuild_df(CONT_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['time_variant'] = 'baseline'

bucket_subset = bucket_df[bucket_df['run_name'].isin(bucket_runs)].copy()
bucket_subset['time_variant'] = 'delta_start_b9'

continuous_subset = continuous_df[continuous_df['run_name'].isin(continuous_runs)].copy()
continuous_subset['time_variant'] = 'delta_start_continuous'

df_ndcg10 = pd.concat([baseline_subset, bucket_subset, continuous_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'time_encoding', 'time_delta_column', 'time_bucket_boundaries_parsed', 'time_feature_dim',
    'best_valid_full_ndcg@10', 'best_valid_full_ndcg@5', 'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_ndcg@5', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_ndcg@5', 'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_ndcg@5', 'best_test_sampled_mrr',
]]


In [ ]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether sampled and MRR move in the same direction
- this notebook compares `delta_start_seconds + 9-bucket` vs `delta_start_seconds + continuous` under the same baseline


## NDCG@5 comparison summary


In [ ]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
bucket_runs = [
    'timeaware_dstart_refine_ml50_do035_b9_s42',
    'timeaware_dstart_refine_ml50_do035_b9_s2024',
    'timeaware_dstart_refine_ml50_do035_b9_s7',
]
continuous_runs = [
    'timeaware_dstart_conti_refine_ml50_do035_s42',
    'timeaware_dstart_conti_refine_ml50_do035_s2024',
    'timeaware_dstart_conti_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
bucket_df = rebuild_df(BUCKET_NDCG5_OUTPUT_DIR)
continuous_df = rebuild_df(CONT_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['time_variant'] = 'baseline'

bucket_subset = bucket_df[bucket_df['run_name'].isin(bucket_runs)].copy()
bucket_subset['time_variant'] = 'delta_start_b9'

continuous_subset = continuous_df[continuous_df['run_name'].isin(continuous_runs)].copy()
continuous_subset['time_variant'] = 'delta_start_continuous'

df_ndcg5 = pd.concat([baseline_subset, bucket_subset, continuous_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'time_encoding', 'time_delta_column', 'time_bucket_boundaries_parsed', 'time_feature_dim',
    'best_valid_full_ndcg@10', 'best_valid_full_ndcg@5', 'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_ndcg@5', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_ndcg@5', 'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_ndcg@5', 'best_test_sampled_mrr',
]]


In [ ]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether sampled and MRR move in the same direction
- this notebook compares `delta_start_seconds + 9-bucket` vs `delta_start_seconds + continuous` under the same baseline
